## Install and imports

In [20]:
#pip install scikit-learn

In [ ]:
import os
import pandas as pd
import sklearn
from sklearn.metrics import mean_absolute_error
import numpy as np

## Round 1

### Route to the csv file and read it

In [ ]:
#Route to the file

BASE_DIR = os.getcwd()

file_path = os.path.join(
    BASE_DIR,
    "Pred_obs_R1.csv",
)

#Read the CSV file
DE_round1 = pd.read_csv(file_path, sep=";")
DE_round1 = DE_round1.dropna(how="all") # Remove rows where all values are NaN

### Metrics calculations (without "noise")

In [ ]:
#MAE calculation 
##Calculate MAE for each variable and store in a dictionary
mae_results1 =  []

for var in variables:
    mae = mean_absolute_error(DE_round1[f"{var}_obs"], DE_round1[f"{var}_pred"])
    mae_results1.append([var, mae])

mae_df1 = pd.DataFrame(mae_results1, columns=["Variable", "MAE"])
mae_df1 = mae_df1.round(3)

#MRE calculation
mre_results1 = []

for var in variables:
    
    y_obs = pd.to_numeric(DE_round1[f"{var}_obs"], errors='coerce')
    y_pred = pd.to_numeric(DE_round1[f"{var}_pred"], errors='coerce')

    df_temp = pd.DataFrame({"obs": y_obs, "pred": y_pred}).dropna()

    y_obs = df_temp["obs"].values
    y_pred = df_temp["pred"].values

    epsilon = 1e-8
    
    mre = np.mean(np.abs((y_obs - y_pred) / (np.abs(y_obs) + epsilon))) * 100

    mre_results1.append([var, mre])

mre_df1 = pd.DataFrame(mre_results1, columns=["Variable", "MRE_%"])
print("\nMRE sin ruido:")



#### Show and print results

In [ ]:
# Show results
print("\nUpload data:")
print(DE_round1.head())

print("\nMetrics without noise:")
print(mae_df1)
print(mre_df1)

# Save results
output_path = os.path.join(BASE_DIR, "Metrics_wo_noiseR1.csv")
mae_df1.to_csv(output_path, index=False)


Upload data:
   Run  Soluplus_mg_ml  TPGS_mg_ml  DTX_mg_ml  Size_pred  Size_obs  PdI_pred  \
0  1.0            50.0        0.05       2.25    121.419     65.38  0.102280   
1  2.0            50.0        0.05       2.25    121.419     65.00  0.102280   
2  3.0            50.0        0.33       2.30    126.329     65.03  0.102865   
3  4.0            50.0        0.33       2.30    126.329     65.04  0.102865   
4  5.0            50.0        0.05       2.50    123.408     64.92  0.104288   

   PdI_obs   ZP_pred  ZP_obs  EE_pred  EE_obs  DL_pred  DL_obs  
0     0.08  0.501265   -0.57    109.7    74.1     11.2     3.2  
1     0.08  0.501265   -0.40    109.7    58.7     11.2     2.5  
2     0.07  0.495279   -0.41    109.4    82.1     11.5     3.6  
3     0.08  0.495279   -0.35    109.4    71.7     11.5     3.1  
4     0.07  0.500015   -0.59    109.5    72.0     13.0     3.4  

MAE without noise:
  Variable     MAE
0     Size  61.644
1      PdI   0.040
2       ZP   0.768
3       EE  32.120


### Metrics calculations (with "noise" *Monte Carlo*)

In [17]:
n_iterations = 200 # numbner of iterations (simulations) for each noise level 

variables = ['Size', 'PdI', 'ZP', 'EE', 'DL']

sigmas = [0.02, 0.05, 0.1] # Noise levels as a percentage values (e.g., 2%, 5%, 10%)
mae_noise_results = []

In [ ]:
epsilon = 1e-8

for sigma_factor in sigmas:
    
    for var in variables:

        #Convert data to numeric and drop NaN values  
        y_obs = pd.to_numeric(DE_round1[f"{var}_obs"], errors='coerce')
        y_pred = pd.to_numeric(DE_round1[f"{var}_pred"], errors='coerce')

        df_temp = pd.DataFrame({"obs": y_obs, "pred": y_pred}).dropna()

        y_obs = df_temp["obs"].values
        y_pred = df_temp["pred"].values

        sigma = sigma_factor * np.mean(np.abs(y_obs))

        mae_list = []
        mre_list = [] 

        #Monte Carlo simulation
        for i in range(n_iterations):

            ruido = np.random.normal(0, sigma, size=len(y_obs))
            y_ruidoso = y_obs + ruido

            ma e = mean_absolute_error(y_ruidoso, y_pred)
            mae_list.append(mae)

            # MRE
            mre = np.mean(np.abs((y_ruidoso - y_pred) / (np.abs(y_ruidoso) + epsilon))) * 100
            mre_list.append(mre)

        mae_noise_results.append([
            var, #analyzed variable
            sigma_factor, #noise level
            np.mean(mae_list), #Mean MAE
            np.std(mae_list) #Standard deviation of MAE
        ])

        mre_noise_results.append([
            var, #analyzed variable
            sigma_factor, #noise level
            np.mean(mre_list), #Mean MRE
            np.std(mre_list)  #Standard deviation of MRE
        ])


#### Show and print results

In [ ]:
mae_noise_df = pd.DataFrame(
    mae_noise_results,
    columns=["Variable", "Sigma_%", "MAE_mean", "MAE_std"]
)

print("\nMetrics with noise (different levels):")
print(mae_noise_df)

output_path = os.path.join(BASE_DIR, "Metrics_noise_R1.csv")
mae_noise_df.to_csv(output_path, index=False)


MAE with noise (different levels):
   Variable  Sigma_%   MAE_mean   MAE_std
0      Size     0.02  61.595150  0.477551
1       PdI     0.02   0.039996  0.000637
2        ZP     0.02   0.767990  0.002137
3        EE     0.02  32.169048  0.452867
4        DL     0.02   7.318084  0.019651
5      Size     0.05  61.654526  1.221124
6       PdI     0.05   0.039731  0.001763
7        ZP     0.05   0.768421  0.005796
8        EE     0.05  32.135180  1.209239
9        DL     0.05   7.316576  0.046906
10     Size     0.10  61.657410  2.309701
11      PdI     0.10   0.039997  0.003616
12       ZP     0.10   0.768141  0.012195
13       EE     0.10  32.424497  2.187155
14       DL     0.10   7.323093  0.097094
15     Size     0.02  61.569739  0.478351
16      PdI     0.02   0.039834  0.000677
17       ZP     0.02   0.768082  0.002509
18       EE     0.02  32.100818  0.510434
19       DL     0.02   7.322043  0.018246
20     Size     0.05  61.642390  1.239701
21      PdI     0.05   0.039954  0.00170

## Round 2

### Route to the csv file and read it

In [35]:
#Route to the file

BASE_DIR = os.getcwd()

file_path = os.path.join(
    BASE_DIR,
    "Pred_obs_R2.csv",
)

#Read the CSV file
DE_round2 = pd.read_csv(file_path, sep=";")
DE_round2 = DE_round2.dropna(how="all") # Remove rows where all values are NaN

### MAE calculations

In [37]:
#Calculate MAE for each variable and store in a dictionary
mae_results2 =  []

for var in variables:
    mae = mean_absolute_error(DE_round2[f"{var}_obs"], DE_round2[f"{var}_pred"])
    mae_results2.append([var, mae])

mae_df2 = pd.DataFrame(mae_results2, columns=["Variable", "MAE"])
mae_df2 = mae_df2.round(3)


### Show and print results

In [38]:
# Show results
print("\nUpload data:")
print(DE_round1.head())

print("\nMAE:")
print(mae_df2)

# Save results
output_path = os.path.join(BASE_DIR, "MAE_results_R2.csv")
mae_df2.to_csv(output_path, index=False)


Upload data:
   Run  Soluplus_mg_ml  TPGS_mg_ml  DTX_mg_ml  Size_pred  Size_obs  PdI_pred  \
0  1.0            50.0        0.05       2.25    121.419     65.38  0.102280   
1  2.0            50.0        0.05       2.25    121.419     65.00  0.102280   
2  3.0            50.0        0.33       2.30    126.329     65.03  0.102865   
3  4.0            50.0        0.33       2.30    126.329     65.04  0.102865   
4  5.0            50.0        0.05       2.50    123.408     64.92  0.104288   

   PdI_obs   ZP_pred  ZP_obs  EE_pred  EE_obs  DL_pred  DL_obs  
0     0.08  0.501265   -0.57    109.7    74.1     11.2     3.2  
1     0.08  0.501265   -0.40    109.7    58.7     11.2     2.5  
2     0.07  0.495279   -0.41    109.4    82.1     11.5     3.6  
3     0.08  0.495279   -0.35    109.4    71.7     11.5     3.1  
4     0.07  0.500015   -0.59    109.5    72.0     13.0     3.4  

MAE:
  Variable     MAE
0     Size  65.680
1      PdI   0.116
2       ZP   0.804
3       EE  55.100
4       DL  11